In [2]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Then import everything else and run training

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split


In [3]:
misplaced_priorities_data_input_path = "data/outputs/mandate_deviation/combined_llm_alignment_results_1_891.csv"
use_columns = [
    'code',
    'ergp_line_item',
    'mda_code',
    'agency',
    'amount',
    'alignment',
    'reason',
]
df = pd.read_csv(misplaced_priorities_data_input_path, usecols=use_columns)


In [4]:
df.head()

,code,ergp_line_item,mda_code,agency,amount,alignment,reason
0,ERGP1102191,IT INFRASTRUCTURE AND ASSOCIATED EQUIPMENT\n,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),9.817500e+07,YES,IT infrastructure and associated equipment dir...
1,ERGP1102209,PURCHASE OF OFFICE FURNITURE AND FITTINGS FOR ...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.490738e+07,YES,Office furniture and fittings are necessary ad...
2,ERGP1205869,PURCHASE OF LAP TOP AND DESKTOP COMPUTERS FOR ...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.107260e+08,YES,Purchasing computers for staff official use is...
3,ERGP1205905,REMODELING OF SECURITY HOUSE AND BPE MAIN ENTR...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.400000e+07,PARTIAL,While the remodeling of BPE's main entrance do...
4,ERGP16234836,"""ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT H...",111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.400000e+10,NO,The budget item refers to acquiring headquarte...


In [5]:
# Check class distribution for 'col2'
class_distribution = df['alignment'].value_counts()

# Print the class distribution
print(class_distribution)

alignment
YES        9775
NO         7564
PARTIAL    3050
ERROR         2
Name: count, dtype: int64


In [6]:
# Filter the DataFrame to remove rows where col2 is 'test1' or 'test2'
filtered_df = df[~df['alignment'].isin(['PARTIAL', 'ERROR'])]

# # Display the filtered DataFrame
# print(filtered_df)

In [7]:
# Check class distribution 
class_distribution = filtered_df['alignment'].value_counts()

# Print the class distribution
print(class_distribution)
print(len(filtered_df))


alignment
YES    9775
NO     7564
Name: count, dtype: int64
17339


In [ ]:
# Simpler approach
train_df, test_df = train_test_split(filtered_df, test_size=0.25, stratify=filtered_df['label'])

# Check coverage
print(f"Train agencies: {train_df['mda_code'].nunique()}/546")
print(f"Test agencies: {test_df['mda_code'].nunique()}/546")

# Likely result: ~780 in train, ~750 in test (sparse agencies missing from test)

Train agencies: 543/546
Test agencies: 489/546


In [8]:
# Get the first 1000 rows
filtered_df = filtered_df[:8000]

# Display the new DataFrame
print(len(filtered_df))

8000


In [21]:
# Simpler approach
train_df, test_df = train_test_split(filtered_df, test_size=0.25, stratify=filtered_df['alignment'])

# Check coverage
print(f"Train agencies: {train_df['mda_code'].nunique()}")
print(f"Test agencies: {test_df['mda_code'].nunique()}")

# Likely result: ~780 in train, ~750 in test (sparse agencies missing from test)

Train agencies: 252
Test agencies: 220


In [9]:
# Check class distribution for 'col2'
class_distribution = filtered_df['alignment'].value_counts()

# Print the class distribution
print(class_distribution)
print(len(filtered_df))


alignment
YES    4357
NO     3643
Name: count, dtype: int64
8000


In [10]:
filtered_df.columns.to_list()

['code',
 'ergp_line_item',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason']

In [11]:
master_mandate_path = "data/inputs/master_combined_mandate_1_to_891.csv"  # actually there are 876?


mandate_df = pd.read_csv(master_mandate_path)
# Rename the 3rd column
mandate_df.columns.values[2] = 'mandate'

In [12]:
mandate_df.columns.to_list()


['mda_code', 'mda_name', 'mandate']

In [13]:
mandate_dict = dict(zip(mandate_df['mda_code'], mandate_df['mandate']))
print(mandate_dict)

{521025001: "The Community Health Tutor Programme at the University College Hospital (UCH), Ibadan is an agency school established within the Federal Training Centre for Teachers of Health Sciences to provide quality teacher-education for community health practitioners in Nigeria. The Programme operates under the supervision of the Federal Ministry of Health as part of the broader initiative to strengthen primary healthcare delivery systems across Nigeria.\n\nThe establishment of tutor programmes at UCH traces its origins to 1976, when the Federal Ministry of Health called upon UCH to assist in training tutors for Schools of Health Technology created across Nigeria to implement primary health care policies. The Federal Government recognized the critical shortage of qualified tutors to teach at these institutions and decided to establish an integrated training centre for teachers of health sciences at UCH, Ibadan.\n\nThe Community Health Officers' Training Programme commenced on October

In [14]:
filtered_df['mandate']= filtered_df['mda_code'].map(mandate_dict)

In [15]:
filtered_df.columns.to_list()


['code',
 'ergp_line_item',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason',
 'mandate']

In [17]:
filtered_df.head()

,code,ergp_line_item,mda_code,agency,amount,alignment,reason,mandate
0,ERGP1102191,IT INFRASTRUCTURE AND ASSOCIATED EQUIPMENT\n,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),9.817500e+07,YES,IT infrastructure and associated equipment dir...,The Bureau of Public Enterprises (BPE) was est...
1,ERGP1102209,PURCHASE OF OFFICE FURNITURE AND FITTINGS FOR ...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.490738e+07,YES,Office furniture and fittings are necessary ad...,The Bureau of Public Enterprises (BPE) was est...
2,ERGP1205869,PURCHASE OF LAP TOP AND DESKTOP COMPUTERS FOR ...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.107260e+08,YES,Purchasing computers for staff official use is...,The Bureau of Public Enterprises (BPE) was est...
4,ERGP16234836,"""ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT H...",111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.400000e+10,NO,The budget item refers to acquiring headquarte...,The Bureau of Public Enterprises (BPE) was est...
6,ERGP15225464,DIRECT PURCHASE OF OPERATIONAL VEHICLES,111010001,BUREAU OF PUBLIC PROCUREMENT (BPP),3.360000e+08,NO,The Bureau of Public Procurement is a regulato...,The Bureau of Public Procurement (BPP) was est...


In [49]:
# filtered_df.columns.values[1] = 'project'


filtered_df = filtered_df.rename(columns={'ergp_line_item': 'project'})


In [56]:
filtered_df.columns.to_list()


['code',
 'project',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason',
 'mandate']

In [57]:
#  create label

alignment_map = {'YES': 1, 'NO': 0}

filtered_df['label'] = filtered_df['alignment'].map(alignment_map)

In [58]:
filtered_df.columns.to_list()


['code',
 'project',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason',
 'mandate',
 'label']

In [59]:
filtered_df['label'].value_counts()


label
1    4357
0    3643
Name: count, dtype: int64

In [77]:
filtered_df.to_csv("trial_budget_dataset_8000_projects.csv", index=False)

In [4]:
import pandas as pd

df = pd.read_csv("trial_budget_dataset_8000_projects.csv")

In [5]:
df.columns.to_list()

['code',
 'project',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason',
 'mandate',
 'label']

In [60]:
df = filtered_df.copy()

In [6]:
train_indices = []
test_indices = []

for agency in df['mda_code'].unique():
    agency_data = df[df['mda_code'] == agency]
    
    if len(agency_data) < 4:
        # Too few projects - all to training
        train_indices.extend(agency_data.index.tolist())
    else:
        # Check if stratification is viable
        can_stratify = (
            len(agency_data) >= 8 and 
            agency_data['alignment'].value_counts().min() >= 2
        )
        
        train_agency, test_agency = train_test_split(
            agency_data,
            test_size=0.25,
            random_state=42,
            stratify=agency_data['alignment'] if can_stratify else None
        )
        train_indices.extend(train_agency.index.tolist())
        test_indices.extend(test_agency.index.tolist())

# train_df = df.loc[train_indices]
train_df = df.loc[train_indices].reset_index(drop=True)
temp_test_df = df.loc[test_indices]
# temp_test_df = df.loc[test_indices].reset_index(drop=True)


print(f"Train: {len(train_df)} projects, {train_df['mda_code'].nunique()} agencies")
print(f"Test: {len(temp_test_df)} projects, {temp_test_df['mda_code'].nunique()} agencies")
print(f"\nTrain alignment:\n{train_df['alignment'].value_counts()}")
print(f"\nTest alignment:\n{temp_test_df['alignment'].value_counts()}")

Train: 5921 projects, 252 agencies
Test: 2079 projects, 238 agencies

Train alignment:
alignment
YES    3202
NO     2719
Name: count, dtype: int64

Test alignment:
alignment
YES    1155
NO      924
Name: count, dtype: int64


In [7]:
test_indicies = []
val_indicies = []

for agency in temp_test_df['mda_code'].unique():
    agency_data = temp_test_df[temp_test_df['mda_code'] == agency]
    
    if len(agency_data) < 4:
        # Too few projects - all to training
        test_indicies.extend(agency_data.index.tolist())
    else:
        # Check if stratification is viable
        can_stratify = (
            len(agency_data) >= 8 and 
            agency_data['alignment'].value_counts().min() >= 2
        )
        
        train_agency, test_agency = train_test_split(
            agency_data,
            test_size=0.40,
            random_state=42,
            stratify=agency_data['alignment'] if can_stratify else None
        )
        test_indicies.extend(train_agency.index.tolist())
        val_indicies.extend(test_agency.index.tolist())

val_df = df.loc[test_indicies]
val_df = df.loc [test_indicies].reset_index(drop=True)
test_df = df.loc[val_indicies]
test_df = df.loc[val_indicies].reset_index(drop=True)


print(f"Validation Set: {len(val_df)} projects, {val_df['mda_code'].nunique()} agencies")
print(f"Test Set: {len(test_df)} projects, {test_df['mda_code'].nunique()} agencies")
# print(f"\nTrain alignment:\n{train_df['alignment'].value_counts()}")
print(f"\nValidation alignment:\n{val_df['alignment'].value_counts()}")
print(f"\nTest alignment:\n{test_df['alignment'].value_counts()}")

Validation Set: 1343 projects, 238 agencies
Test Set: 736 projects, 93 agencies

Validation alignment:
alignment
YES    788
NO     555
Name: count, dtype: int64

Test alignment:
alignment
NO     369
YES    367
Name: count, dtype: int64


In [8]:
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Train: 5921 | Val: 1343 | Test: 736


In [9]:
train_df.shape

(5921, 9)

In [10]:
train_df.columns.to_list()

['code',
 'project',
 'mda_code',
 'agency',
 'amount',
 'alignment',
 'reason',
 'mandate',
 'label']

In [11]:
train_df.head()

,code,project,mda_code,agency,amount,alignment,reason,mandate,label
0,ERGP16234836,"""ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT H...",111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.400000e+10,NO,The budget item refers to acquiring headquarte...,The Bureau of Public Enterprises (BPE) was est...,0
1,ERGP1102191,IT INFRASTRUCTURE AND ASSOCIATED EQUIPMENT\n,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),9.817500e+07,YES,IT infrastructure and associated equipment dir...,The Bureau of Public Enterprises (BPE) was est...,1
2,ERGP1205869,PURCHASE OF LAP TOP AND DESKTOP COMPUTERS FOR ...,111007001,BUREAU OF PUBLIC ENTERPRISES (BPE),1.107260e+08,YES,Purchasing computers for staff official use is...,The Bureau of Public Enterprises (BPE) was est...,1
3,ERGP15225564,PROCUREMENT OF PHOTOCOPIER MACHINE,111010001,BUREAU OF PUBLIC PROCUREMENT (BPP),3.150000e+07,NO,The Bureau of Public Procurement regulates and...,The Bureau of Public Procurement (BPP) was est...,0
4,ERGP15225657,PROCUREMENT AUDIT,111010001,BUREAU OF PUBLIC PROCUREMENT (BPP),1.400000e+08,YES,Procurement audit directly aligns with BPP's m...,The Bureau of Public Procurement (BPP) was est...,1


In [ ]:
# # Split data

# from sklearn.model_selection import train_test_split

# # Stratified split (maintains class balance)
# train_df, temp_df = train_test_split(
#     df, 
#     test_size=0.25, 
#     random_state=42, 
#     stratify=df['alignment']
# )

# val_df, test_df = train_test_split(
#     temp_df, 
#     test_size=0.60,  # 60% of 25% = 15% of total
#     random_state=42, 
#     stratify=temp_df['alignment']
# )

# print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
# # Train: 37500 | Val: 7500 | Test: 5000

In [12]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import math

# ============================================
# 1. PREPARE DATA
# ============================================

# Convert to InputExample format
train_examples = []
for idx, row in train_df.iterrows():
    train_examples.append(
        InputExample(
            texts=[row['mandate'], row['project']], 
            label=int(row['label'])
        )
    )

val_examples = []
for idx, row in val_df.iterrows():
    val_examples.append(
        InputExample(
            texts=[row['mandate'], row['project']], 
            label=int(row['label'])
        )
    )


/Users/admin/PycharmProjects/fg_interactive_budget/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from sentence_transformers import CrossEncoder



# # ============================================
# # 2. LOAD BASE MODEL
# # ============================================

model = CrossEncoder(
    'cross-encoder/nli-deberta-v3-large',
    num_labels=2,
    max_length=1024,
    automodel_args={'ignore_mismatched_sizes': True},
    # device='cpu'
)


The CrossEncoder `automodel_args` argument was renamed and is now deprecated, please use `model_kwargs` instead.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at cross-encoder/nli-deberta-v3-large and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([2, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# ============================================
# 3. CONFIGURE TRAINING
# ============================================



train_dataloader = DataLoader(
    train_examples, 
    shuffle=True, 
    batch_size=16 # Adjust based on GPU memory
)

In [17]:
# ============================================
# 4. TRAIN
# ============================================

num_epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% warmup

model.fit(
    train_dataloader=train_dataloader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path='./models/deberta-v3-budget-classifier',
    save_best_model=True,
    show_progress_bar=True
)

print("Training complete!")

# After training
model.save('./models/deberta-v3-budget-classifier')

RuntimeError: MPS backend out of memory (MPS allocated: 2.80 GB, other allocations: 598.15 MB, max allowed: 3.40 GB). Tried to allocate 57.19 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# # Step 2: Fine-Tuning with Sentence-Transformers (Recommended)
# # Method A: Simple Fine-Tuning

# from sentence_transformers import CrossEncoder, InputExample
# from torch.utils.data import DataLoader
# import math

# # ============================================
# # 1. PREPARE DATA
# # ============================================

# # Convert to InputExample format
# train_examples = []
# for idx, row in train_df.iterrows():
#     train_examples.append(
#         InputExample(
#             texts=[row['mandate'], row['project']], 
#             label=int(row['label'])
#         )
#     )

# val_examples = []
# for idx, row in val_df.iterrows():
#     val_examples.append(
#         InputExample(
#             texts=[row['mandate'], row['project']], 
#             label=int(row['label'])
#         )
#     )

# # ============================================
# # 2. LOAD BASE MODEL
# # ============================================

# model = CrossEncoder(
#     'cross-encoder/nli-deberta-v3-large',  # 1024 token limit
#     num_labels=2,
#     max_length=1024  # Explicitly set max length
# )

# # ============================================
# # 3. CONFIGURE TRAINING
# # ============================================

# train_dataloader = DataLoader(
#     train_examples, 
#     shuffle=True, 
#     batch_size=16  # Adjust based on GPU memory
# )

# # ============================================
# # 4. TRAIN
# # ============================================

# num_epochs = 3
# warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% warmup

# model.fit(
#     train_dataloader=train_dataloader,
#     epochs=num_epochs,
#     warmup_steps=warmup_steps,
#     output_path='./models/deberta-v3-budget-classifier',
#     save_best_model=True,
#     show_progress_bar=True
# )

# print("Training complete!")

In [ ]:
# Option 1: Basic Evaluation 

from sentence_transformers import CrossEncoder
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ============================================
# LOAD FINE-TUNED MODEL
# ============================================

model = CrossEncoder('./models/deberta-v3-budget-classifier')

# ============================================
# PREPARE TEST DATA
# ============================================

test_pairs = [
    (row['mandate'], row['project']) 
    for idx, row in test_df.iterrows()
]
test_labels = test_df['label'].values

# ============================================
# PREDICT
# ============================================

# Get logits
predictions_logits = model.predict(test_pairs, show_progress_bar=True)

# Convert to probabilities
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)

predictions_probs = softmax(predictions_logits)

# Get predicted classes
predictions = np.argmax(predictions_logits, axis=1)

# ============================================
# EVALUATION METRICS
# ============================================

print(classification_report(
    test_labels, 
    predictions, 
    target_names=['Within Mandate (0)', 'Outside Mandate (1)']
))

# Output:
#                        precision    recall  f1-score   support
# Within Mandate (0)         0.92      0.95      0.93      4200
# Outside Mandate (1)        0.78      0.68      0.73       800
#           accuracy                             0.90      5000

In [ ]:
# Option 2: Comprehensive Evaluation 


from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# COMPUTE METRICS
# ============================================

def evaluate_model(y_true, y_pred, y_probs):
    """Comprehensive evaluation"""
    
    # Basic metrics
    acc = accuracy_score(y_true, y_pred)
    
    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None
    )
    
    # Weighted averages
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )
    
    # Class 1 specific (outside mandate - what you care about)
    precision_class1 = precision[1]
    recall_class1 = recall[1]
    f1_class1 = f1[1]
    
    # False positive rate
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)
    
    # AUC
    auc = roc_auc_score(y_true, y_probs[:, 1])
    
    # Print results
    print("=" * 60)
    print("OVERALL METRICS")
    print("=" * 60)
    print(f"Accuracy:           {acc:.4f}")
    print(f"F1 (weighted):      {f1_weighted:.4f}")
    print(f"Precision (weighted): {precision_weighted:.4f}")
    print(f"Recall (weighted):    {recall_weighted:.4f}")
    print(f"AUC-ROC:            {auc:.4f}")
    print()
    
    print("=" * 60)
    print("CLASS 0: Within Mandate")
    print("=" * 60)
    print(f"Precision:          {precision[0]:.4f}")
    print(f"Recall:             {recall[0]:.4f}")
    print(f"F1-Score:           {f1[0]:.4f}")
    print(f"Support:            {support[0]}")
    print()
    
    print("=" * 60)
    print("CLASS 1: Outside Mandate (Critical for Your Task)")
    print("=" * 60)
    print(f"Precision:          {precision_class1:.4f}  ← How often flags are correct")
    print(f"Recall:             {recall_class1:.4f}  ← % of violations caught")
    print(f"F1-Score:           {f1_class1:.4f}")
    print(f"Support:            {support[1]}")
    print()
    
    print("=" * 60)
    print("ERROR ANALYSIS")
    print("=" * 60)
    print(f"False Positive Rate: {fpr:.4f}  ← % good projects wrongly flagged")
    print(f"False Positives:     {fp} / {fp + tn} Class 0 examples")
    print(f"False Negatives:     {fn} / {fn + tp} Class 1 examples")
    print()
    
    return {
        'accuracy': acc,
        'f1': f1_weighted,
        'precision_class1': precision_class1,
        'recall_class1': recall_class1,
        'f1_class1': f1_class1,
        'fpr': fpr,
        'auc': auc,
        'confusion_matrix': cm
    }

# Run evaluation
metrics = evaluate_model(test_labels, predictions, predictions_probs)

In [ ]:
# Option 3: Visualisation 

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# 1. CONFUSION MATRIX
# ============================================

cm = confusion_matrix(test_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['Within (0)', 'Outside (1)'],
    yticklabels=['Within (0)', 'Outside (1)']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# ============================================
# 2. ROC CURVE
# ============================================

fpr, tpr, thresholds = roc_curve(test_labels, predictions_probs[:, 1])
auc = roc_auc_score(test_labels, predictions_probs[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.show()

# ============================================
# 3. CONFIDENCE DISTRIBUTION
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class 0 confidence
class0_mask = test_labels == 0
class0_confidence = predictions_probs[class0_mask, 0]

axes[0].hist(class0_confidence, bins=50, color='green', alpha=0.7)
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Class 0 (Within Mandate) - Confidence Distribution')
axes[0].axvline(0.7, color='red', linestyle='--', label='Threshold')
axes[0].legend()

# Class 1 confidence
class1_mask = test_labels == 1
class1_confidence = predictions_probs[class1_mask, 1]

axes[1].hist(class1_confidence, bins=50, color='orange', alpha=0.7)
axes[1].set_xlabel('Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Class 1 (Outside Mandate) - Confidence Distribution')
axes[1].axvline(0.7, color='red', linestyle='--', label='Threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=300)
plt.show()

# ============================================
# 4. PRECISION-RECALL CURVE
# ============================================

from sklearn.metrics import precision_recall_curve, average_precision_score

precision_curve, recall_curve, thresholds_pr = precision_recall_curve(
    test_labels, predictions_probs[:, 1]
)
avg_precision = average_precision_score(test_labels, predictions_probs[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve, label=f'AP = {avg_precision:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Class 1: Outside Mandate)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=300)
plt.show()

In [ ]:
# ============================================
# ANALYZE MISCLASSIFICATIONS
# ============================================

test_df_copy = test_df.copy()
test_df_copy['predicted'] = predictions
test_df_copy['confidence_class0'] = predictions_probs[:, 0]
test_df_copy['confidence_class1'] = predictions_probs[:, 1]
test_df_copy['correct'] = test_df_copy['label'] == test_df_copy['predicted']

# False positives (flagged as outside, but actually within)
false_positives = test_df_copy[
    (test_df_copy['label'] == 0) & (test_df_copy['predicted'] == 1)
].sort_values('confidence_class1', ascending=False)

print("=" * 60)
print("TOP 10 FALSE POSITIVES (Wrongly Flagged)")
print("=" * 60)
for idx, row in false_positives.head(10).iterrows():
    print(f"\nMandate: {row['mandate'][:100]}...")
    print(f"Project: {row['project'][:100]}...")
    print(f"Confidence: {row['confidence_class1']:.2%}")
    print("-" * 60)

# False negatives (should be flagged, but missed)
false_negatives = test_df_copy[
    (test_df_copy['label'] == 1) & (test_df_copy['predicted'] == 0)
].sort_values('confidence_class0', ascending=False)

print("\n" + "=" * 60)
print("TOP 10 FALSE NEGATIVES (Missed Violations)")
print("=" * 60)
for idx, row in false_negatives.head(10).iterrows():
    print(f"\nMandate: {row['mandate'][:100]}...")
    print(f"Project: {row['project'][:100]}...")
    print(f"Confidence: {row['confidence_class0']:.2%}")
    print("-" * 60)

# Save full error analysis
test_df_copy.to_csv('test_results_with_predictions.csv', index=False)

In [ ]:
# ============================================
# EVALUATE BY AGENCY
# ============================================

# Assuming you have MDA codes in your test data
test_df_copy['mda_code'] = test_df['mda_code']  # Add if available

agency_performance = test_df_copy.groupby('mda_code').apply(
    lambda group: pd.Series({
        'total': len(group),
        'accuracy': (group['label'] == group['predicted']).mean(),
        'class1_count': (group['label'] == 1).sum(),
        'class1_recall': (
            (group['label'] == 1) & (group['predicted'] == 1)
        ).sum() / max((group['label'] == 1).sum(), 1)
    })
).sort_values('accuracy')

print("Worst Performing Agencies:")
print(agency_performance.head(10))

print("\nBest Performing Agencies:")
print(agency_performance.tail(10))

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

# ============================================
# LOAD FINE-TUNED MODEL
# ============================================

model = CrossEncoder('./models/deberta-v3-budget-classifier')

# ============================================
# INFERENCE ON 2026 BUDGET
# ============================================

# Load your 2026 budget (20,000 projects)
budget_2026 = pd.read_csv('budget_2026.csv')

# Prepare pairs
pairs = [
    (row['mandate'], row['project']) 
    for idx, row in budget_2026.iterrows()
]

# Batch predict
batch_size = 32
all_predictions = []

for i in range(0, len(pairs), batch_size):
    batch = pairs[i:i+batch_size]
    preds = model.predict(batch)
    all_predictions.extend(preds)

all_predictions = np.array(all_predictions)

# Softmax for probabilities
probs = np.exp(all_predictions) / np.exp(all_predictions).sum(axis=1, keepdims=True)

# Add results to dataframe
budget_2026['predicted_class'] = np.argmax(all_predictions, axis=1)
budget_2026['prob_within'] = probs[:, 0]
budget_2026['prob_outside'] = probs[:, 1]

# Flag high-confidence violations
budget_2026['flag_for_review'] = (
    (budget_2026['predicted_class'] == 1) & 
    (budget_2026['prob_outside'] > 0.7)
)

print(f"Total projects flagged: {budget_2026['flag_for_review'].sum()}")
print(f"Percentage flagged: {budget_2026['flag_for_review'].mean():.2%}")

# Export results
budget_2026.to_csv('budget_2026_classified.csv', index=False)